Steps
1. Chunk
2. Embed each chunk with 

In [1]:
from dotenv import load_dotenv
import os
load_dotenv(override=True) 

True

# Prep data - convert PDF to markdown

In [2]:
# unlock the PDF file
import pikepdf
from markitdown import MarkItDown

with pikepdf.open("data/locked_JJ_10K.pdf") as pdf:
    pdf.save("data/unlocked_JJ_10K.pdf")
# convert the PDF to markdown
md = MarkItDown()
results = md.convert("data/unlocked_JJ_10K.pdf")
# save the markdown to a file
with open("data/JJ_10K.md", "w", encoding='utf-8') as f:
    f.write(results.text_content)

# note: Claude parsed markdown better than Markitdown,
# will use claude output for now. need to clean up pdf parsing and markdown conversion in the future.

c:\Users\Josiah\Git_Repos\EasyRag\.venv\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


KeyboardInterrupt: 

# Vectorize Document

In [12]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
import openai
from util import Embedder
import os

In [3]:
class MockUploadedFile:
    def __init__(self, path):
        self.name = path.split("/")[-1]   # or os.path.basename(path)
        self.type = self._get_mime()
    
    def _get_mime(self):
        ext = self.name.split(".")[-1].lower()
        mime_map = {
            "txt":  "text/plain",
            "md":   "text/markdown",
            "pdf":  "application/pdf",
            "docx": "application/vnd.openxmlformats-officedocument.wordprocessingml.document",
        }
        return mime_map.get(ext, "application/octet-stream")
    
    def read(self):
        with open(self._path, "rb") as f:
            return f.read()
    
    def __init__(self, path):
        self._path = path
        self.name = path.split("\\")[-1].split("/")[-1]
        self.type = self._get_mime()

# Usage in notebook
mock_file = MockUploadedFile("./data/example.txt")

In [4]:
Embedder = Embedder()
print('current collections:', Embedder.collection_names)

Initializing Embedder with model text-embedding-3-small
current collections: []


In [ ]:
Embedder.embed_file(mock_file)
print(Embedder.collection_names)

Attempting to embed file: example.txt of type text/plain
Embedding file: example.txt of type text/plain


(True, 'File example.txt embedded successfully.')

In [10]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
import os

# 1. Create embeddings (LangChain style)
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.getenv("OPENAI_API_KEY")
)

# 2. Connect to existing Chroma DB
vectorstore = Chroma(
    persist_directory="./chroma_db",   # same path as before
    embedding_function=Embedder.embedding_model,
    collection_name="example-txt"  # same collection name as before
)

print("Available collections:", vectorstore._client.list_collections())

# 3. Query
results = vectorstore.similarity_search(
    "What is the purpose of this document?",
    k=3
)



Available collections: [Collection(name=langchain), Collection(name=example-txt)]


In [11]:
for r in results:
    print(r.page_content)
    print(20*"---")
    print(20*"---")
    print()

- Janssen filed litigation in July 2023 challenging the IRA's constitutionality under the First and Fifth Amendments; appeal filed to the Third Circuit in April 2024.
- EU regulations: NIS2, EHDS, Data Act, Cyber Resilience Act, AI Act — increasing privacy and cybersecurity compliance requirements.
- China data security and personal information protection regulations.
- 340B Drug Pricing Program requirements.

#### Employees and Human Capital Management

| Metric | 2024 | 2023 |
------------------------------------------------------------
------------------------------------------------------------

- U.S. FDA regulation of pharmaceutical products and medical technology (product safety, efficacy, manufacturing, advertising, labeling, safety reporting)
- Inflation Reduction Act (IRA, 2022): CMS authorized to negotiate Medicare prices for high-spend drugs starting in 2026 (Part D) and 2028 (Part B). XARELTO, STELARA, and IMBRUVICA are on the first Selected Drug list.
--------------------

In [4]:
import chromadb

# Connect to your ChromaDB (adjust path if using persistent storage)
client = chromadb.PersistentClient(path="./chroma_db" )  # or chromadb.Client() for in-memory

# List all collections
print("Collections:", client.list_collections())

# Get a specific collection
try:
    collection = client.get_collection(name="example-txt")
except Exception as e:
    print(f"Error retrieving collection: {e}")
    collection = None

# Check basic info
if collection:
    print("Count:", collection.count())

# Peek at a few items
if collection:
    print(collection.peek(limit=5))

Collections: []
Error retrieving collection: Collection [example-txt] does not exist


In [ ]:
import util
import util.agent
import util.embedder

embedder = util.embedder.Embedder()
agent = util.agent.build_agent()



In [3]:
print(agent.context.history)
agent_response = agent.run("Search the web for a joke?")
print('-----------------------------------')
print(agent.context.history)

[]
User input: Search the web for a joke?
Agent loop iteration 1
Model response: Message(id='msg_011Cdewo35AKChvDaXR9f7rw', container=None, content=[ServerToolUseBlock(id='srvtoolu_01AS5J3NGSX7P1MPSnQMMVjr', caller=None, input={'query': 'funny joke'}, name='web_search', type='server_tool_use'), WebSearchToolResultBlock(caller=DirectCaller(type='direct'), content=[WebSearchResultBlock(encrypted_content='Eo0OCioIEhgCIiRmN2Q5OTQ0My00NmMxLTQxNzEtOTE4MC03NWYxNWNkNmU4NDcSDDT5cIDP0r2JMCdnhBoMJ7PkkVpDaqWV2+imIjCiElHD9I1yxuxslKdKqf+tOCiImfliVB26R+0nanCArM1cJI+Sw2eq+3Foht8WdY4qkA21x/WxQd3gZV+svDkD8y2bMY2JeY3vs76Op6iqnq2gXf49qQb2VzVknik8bcR4zaBMG3pehveXT8zaf5AUwfb/S++CLgzS3/GO67NJnfNvkkYsaeeixaNYBMnwdyqfjah2Dt5uvQj3AFfAl1Hc5ZUumNisIafOcUcuxFn7k3HOJkRiYD+ziiztqx1WHePAwQZWAFrXCi07i1Oq3hNuDoqRf3gFKKhhR6BWPqc0B+Q2BYh0lGbsgpNOUb06oWqiU1ZXVJGiJogIOXGeToPMmkoABFJvCX6LUYoLeZ8w5f0Ox132C3oFrctvU5Qq9nnLbavqUJOdNMdhqw4tFXXIBMnsSq5dEsVeaVX4fwA7Tlp4APRCRDgx7tTDTfWzt5szsRZNAhO6sdl/EMawa3w2u7meHq8ZGHvKuKy+RAJCE7

In [3]:
agent_response

"I am a **helpful research assistant** here to help you find, understand, and synthesize information. Here's what I can do:\n\n1. 📚 **Search Your Knowledge Base** – I can search through your own uploaded documents and files stored in a local database to find relevant passages and answer questions grounded in *your* content.\n\n2. 🌐 **Search the Web** – I can look up current, real-time, or specialized information from the public internet, such as recent news, up-to-date facts, technical documentation, and more.\n\n3. 📝 **Count Words** – I can count the words in any piece of text you provide.\n\n4. 🔍 **Synthesize & Analyze** – I can combine information from multiple sources to give you well-rounded, cited answers to complex questions.\n\n---\n\n**How to get started:**\n- Ask me a question about your documents, and I'll search your knowledge base.\n- Ask me about something current or general, and I'll search the web.\n- Or just have a conversation — I'm happy to help with explanations, su